# 1. Setup: Install dependencies and import libraries

In [1]:
# @title
# ============================================================
# 📦 SETUP
# Install dependencies and import libraries
# ============================================================

!pip install pandas_datareader

import pandas as pd
import yfinance as yf
import requests
import io
from pandas_datareader import data as web
import datetime
import os

# 2. GENERIC DATA DOWNLOAD & CLEANING FUNCTION
# - Downloads price data
# - Extracts Close prices
# - Cleans missing data
# - Returns clean DataFrame

In [11]:
def download_and_clean(tickers, start="2014-01-01", end="2025-12-31"):

    print(f"Downloading {len(tickers)} tickers...")

    data = yf.download(
        tickers,
        start=start,
        end=end,
        auto_adjust=True,
        threads=True
    )

    close_prices = data["Close"]

    # Remove completely empty columns
    df = close_prices.dropna(axis=1, how='all')

    # Keep assets with ≥95% data
    threshold = int(0.95 * len(df))
    df = df.dropna(axis=1, thresh=threshold)

    # Forward fill small gaps
    df = df.ffill()

    # Drop remaining NaNs
    df = df.dropna()

    print(f"Final shape: {df.shape}")

    return df

# 3.1 🇺🇸 S&P 500 — Ticker Extraction + Data Download

In [3]:
def get_sp500_tickers():
    """
    Scrape S&P 500 constituents from Wikipedia
    and format for Yahoo Finance
    """
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    headers = {'User-Agent': 'Mozilla/5.0'}

    response = requests.get(url, headers=headers)
    tables = pd.read_html(response.text)
    df = tables[0]

    tickers = df['Symbol'].str.replace('.', '-', regex=False).tolist()
    return tickers


sp500_tickers = get_sp500_tickers()
sp500_clean = download_and_clean(sp500_tickers)

sp500_clean.to_csv("sp500_close_clean.csv")
print("Saved: sp500_close_clean.csv")

/tmp/ipykernel_3399/1187498659.py:10: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


[*********************100%***********************]  503 of 503 completed
ERROR:yfinance:
2 Failed downloads:
ERROR:yfinance:['SNDK', 'Q']: YFPricesMissingError('possibly delisted; no price data found  (1d 2015-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1420088400, endDate = 1735707600")')


Final shape: (2390, 465)
Saved: sp500_close_clean.csv


# 3.2 🇧🇷 Ibovespa (Brazil)

In [5]:
def get_bovespa_tickers():
    url = "https://en.wikipedia.org/wiki/List_of_companies_listed_on_B3"
    headers = {"User-Agent": "Mozilla/5.0"}

    response = requests.get(url, headers=headers)
    tables = pd.read_html(response.text)

    df = tables[0]
    tickers = df["Ticker"].astype(str).str.strip() + ".SA"

    return tickers.tolist()


bovespa_tickers = get_bovespa_tickers()
bovespa_clean = download_and_clean(bovespa_tickers)

bovespa_clean.to_csv("bovespa_close_clean.csv")
print("Saved: bovespa_close_clean.csv")

/tmp/ipykernel_3399/3999626255.py:10: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


[*                      3%                       ]  3 of 88 completedERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: AZUL4.SA"}}}
[*********************100%***********************]  88 of 88 completed
ERROR:yfinance:
22 Failed downloads:
ERROR:yfinance:['AZUL4.SA', 'CCRO3.SA', 'NTCO3.SA', 'ELET3.SA', 'JBSS3.SA', 'ELET6.SA', 'VIIA3.SA', 'PETZ3.SA', 'LCAM3.SA', 'SULA11.SA', 'CPLE6.SA', 'EMBR3.SA', 'BRFS3.SA', 'BRML3.SA', 'CIEL3.SA', 'CRFB3.SA', 'ENBR3.SA', 'MRFG3.SA', 'BPAN4.SA', 'BIDI11.SA', 'SOMA3.SA', 'GOLL4.SA']: YFTzMissingError('possibly delisted; no timezone found')


Final shape: (2425, 57)
Saved: bovespa_close_clean.csv


# 3.3 🇪🇺 EURO STOXX 50

In [6]:
def get_eurostoxx_tickers():
    url = "https://en.wikipedia.org/wiki/EURO_STOXX_50"
    headers = {"User-Agent": "Mozilla/5.0"}

    response = requests.get(url, headers=headers)
    tables = pd.read_html(io.StringIO(response.text))

    for t in tables:
        if "Ticker" in t.columns or "Symbol" in t.columns:
            col = "Ticker" if "Ticker" in t.columns else "Symbol"
            return t[col].astype(str).str.strip().tolist()


euro_tickers = get_eurostoxx_tickers()
euro_clean = download_and_clean(euro_tickers)

euro_clean.to_csv("eurostoxx50_close_clean.csv")
print("Saved: eurostoxx50_close_clean.csv")

[*********************100%***********************]  50 of 50 completed


Final shape: (2563, 46)
Saved: eurostoxx50_close_clean.csv


# 3.4 🇬🇧 FTSE 100

In [7]:
def get_ftse100_tickers():
    url = "https://en.wikipedia.org/wiki/FTSE_100_Index"
    headers = {"User-Agent": "Mozilla/5.0"}

    response = requests.get(url, headers=headers)
    tables = pd.read_html(io.StringIO(response.text))

    for t in tables:
        if "EPIC" in t.columns or "Ticker" in t.columns:
            col = "EPIC" if "EPIC" in t.columns else "Ticker"
            return (t[col].astype(str).str.strip() + ".L").tolist()


ftse_tickers = get_ftse100_tickers()
ftse_clean = download_and_clean(ftse_tickers)

ftse_clean.to_csv("ftse100_close_clean.csv")
print("Saved: ftse100_close_clean.csv")

[*********************100%***********************]  100 of 100 completed
ERROR:yfinance:
2 Failed downloads:
ERROR:yfinance:['BT.A.L']: YFTzMissingError('possibly delisted; no timezone found')
ERROR:yfinance:['MTLN.L']: YFPricesMissingError('possibly delisted; no price data found  (1d 2015-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1420070400, endDate = 1735689600")')


Final shape: (2472, 91)
Saved: ftse100_close_clean.csv


# 3.5 🇿🇦 JSE Top 40

In [8]:
jse_tickers = [
    "ABG.JO", "AGL.JO", "ANG.JO", "ANH.JO", "APN.JO", "BHG.JO", "BID.JO",
    "BVT.JO", "BTI.JO", "CPI.JO", "CLS.JO", "DSY.JO", "EXX.JO", "FSR.JO",
    "GLN.JO", "GFI.JO", "GRT.JO", "IMP.JO", "INL.JO", "INP.JO", "MNP.JO",
    "MRP.JO", "MTN.JO", "MCG.JO", "NPN.JO", "NED.JO", "NRP.JO", "NPH.JO",
    "OMU.JO", "PRX.JO", "RNI.JO", "REM.JO", "RMH.JO", "SLM.JO", "SOL.JO",
    "SHP.JO", "SBK.JO","SSW.JO", "VOD.JO", "WHL.JO"
]

jse_clean = download_and_clean(jse_tickers)

jse_clean.to_csv("jse40_close_clean.csv")
print("Saved: jse40_close_clean.csv")

[*********************100%***********************]  40 of 40 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['MCG.JO']: YFTzMissingError('possibly delisted; no timezone found')


Final shape: (2527, 31)
Saved: jse40_close_clean.csv


# 🇯🇵 NIKKEI 225 — Japan

In [9]:
nikkei_tickers = [
    "6857.T","8267.T","5201.T","2802.T","6770.T","6113.T","9202.T",
    "8304.T","2502.T","3407.T","4503.T","7832.T","5108.T","7751.T",
    "6952.T","9022.T","9502.T","4519.T","7762.T","1721.T","7186.T",
    "8253.T","4751.T","7912.T","8750.T","4568.T","6367.T","1925.T",
    "8601.T","2432.T","4061.T","6902.T","4324.T","4631.T","5714.T",
    "9020.T","6361.T","4523.T","5020.T","6954.T","9983.T","6504.T",
    "4901.T","5803.T","6702.T","8354.T","5801.T","6674.T","1808.T",
    "7205.T","6305.T","7004.T","6501.T","7267.T","7741.T","5019.T",
    "7013.T","1605.T","3099.T","7202.T","8001.T","3086.T","9201.T",
    "8697.T","6178.T","2914.T","5411.T","1963.T","6473.T","1812.T",
    "4452.T","7012.T","9107.T","9433.T","9008.T","9009.T","6861.T",
    "2801.T","2503.T","5406.T","6301.T","9766.T","4902.T","6326.T",
    "3405.T","6971.T","4151.T","6920.T","4689.T","2413.T","8002.T",
    "8252.T","7261.T","2269.T","4385.T","6479.T","4188.T","8058.T",
    "6503.T","8802.T","7011.T","9301.T","5711.T","7211.T","8306.T",
    "8031.T","4183.T","8801.T","5706.T","9104.T","8411.T","8725.T",
    "6981.T","6701.T","3659.T","5333.T","2282.T","2871.T","6594.T",
    "7731.T","7974.T","5214.T","9147.T","3863.T","5401.T","9432.T",
    "9101.T","4021.T","7201.T","2002.T","1332.T","9843.T","6988.T",
    "8604.T","6471.T","6472.T","9613.T","1802.T","9007.T","3861.T",
    "6103.T","7733.T","6645.T","4661.T","8591.T","9532.T","4578.T",
    "5541.T","6752.T","4755.T","6098.T","6723.T","8308.T","4004.T",
    "7752.T","2501.T","7735.T","9735.T","6724.T","1928.T","3382.T",
    "6753.T","1803.T","4063.T","4507.T","4911.T","5831.T","6273.T",
    "9434.T","9984.T","2768.T","8630.T","6758.T","7270.T","3436.T",
    "4005.T","8053.T","5802.T","6302.T","5713.T","8316.T","8309.T",
    "5232.T","4506.T","8830.T","7269.T","8795.T","5233.T","1801.T",
    "6976.T","2531.T","8233.T","4502.T","6762.T","3401.T","4543.T",
    "8331.T","5631.T","9503.T","5101.T","9001.T","9602.T","5301.T",
    "8766.T","4043.T","9501.T","8035.T","9531.T","8804.T","9005.T",
    "3289.T","7911.T","3402.T","4042.T","5332.T","7203.T","8015.T",
    "4704.T","4208.T","9021.T","7951.T","7272.T","9064.T","6506.T",
    "6841.T"
]

# Download + clean using shared function
nikkei_clean = download_and_clean(nikkei_tickers)

# Save
nikkei_clean.to_csv("nikkei225_close_clean.csv")
print("Saved: nikkei225_close_clean.csv")

[*********************100%***********************]  225 of 225 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['9613.T']: YFTzMissingError('possibly delisted; no timezone found')


Final shape: (2466, 218)
Saved: nikkei225_close_clean.csv


# 🇦🇺 ASX 50 — Australia

In [10]:
asx50_tickers = [
    "CBA.AX","BHP.AX","CSL.AX","NAB.AX","WBC.AX","ANZ.AX","MQG.AX","WES.AX",
    "GMG.AX","WDS.AX","RIO.AX","TCL.AX","WOW.AX","ALL.AX","FMG.AX","STO.AX",
    "BXB.AX","COL.AX","TLS.AX","QBE.AX","ORG.AX","APA.AX","SCG.AX","SHL.AX",
    "S32.AX","IAG.AX","CPU.AX","JHX.AX","NST.AX","RMD.AX","MIN.AX","ALD.AX",
    "BSL.AX","ASX.AX","REA.AX","XRO.AX","WTC.AX","SEK.AX","CAR.AX","QAN.AX",
    "AZJ.AX","LYC.AX","IGO.AX","ILU.AX","ORI.AX","TWE.AX","EDV.AX","TLC.AX",
    "SVW.AX","NWS.AX"
]

# Download + clean using shared function
asx50_clean = download_and_clean(asx50_tickers)

# Save to CSV
asx50_clean.to_csv("asx50_close_clean.csv")
print("Saved: asx50_close_clean.csv")

[*********************100%***********************]  50 of 50 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['SVW.AX']: YFTzMissingError('possibly delisted; no timezone found')


Final shape: (2439, 45)
Saved: asx50_close_clean.csv


#  🌍 MACRO DATA: FX + YIELDS + RISK FACTORS

In [12]:
def get_macro_data_full(start="2014-01-01", end="2025-12-31"):

    # --------------------------------------------------------
    # 📉 1. FX RATES (Cross-currency structure)
    # --------------------------------------------------------
    fx_map = {
        # Major crosses vs USD
        'EURUSD': 'EURUSD=X',
        'GBPUSD': 'GBPUSD=X',
        'JPYUSD': 'JPYUSD=X',
        'AUDUSD': 'AUDUSD=X',
        'BRLUSD': 'BRLUSD=X',
        'ZARUSD': 'ZARUSD=X',

        # Cross relationships (important for relative regimes)
        'EURGBP': 'EURGBP=X',
        'EURJPY': 'EURJPY=X',
        'AUDJPY': 'AUDJPY=X'
    }

    # --------------------------------------------------------
    # 📊 2. RISK FACTORS (Global drivers)
    # --------------------------------------------------------
    risk_map = {
        'VIX': '^VIX',
        'Gold': 'GC=F',
        'Oil': 'CL=F',
        'Bitcoin': 'BTC-USD'
    }

    # Combine Yahoo tickers
    yf_map = {**fx_map, **risk_map}
    yf_tickers = list(yf_map.values())

    print(f"Downloading {len(yf_tickers)} Yahoo series...")

    yf_data = yf.download(yf_tickers, start=start, end=end)["Close"]
    yf_data = yf_data.rename(columns={v: k for k, v in yf_map.items()})

    # --------------------------------------------------------
    # 🏦 3. GOVERNMENT BOND YIELDS (FRED)
    # --------------------------------------------------------
    fred_map = {
        # US
        'US_10Y': 'DGS10',

        # Europe (proxy: Germany)
        'DE_10Y': 'IRLTLT01DEM156N',

        # UK
        'UK_10Y': 'IRLTLT01GBM156N',

        # Japan
        'JP_10Y': 'IRLTLT01JPM156N',

        # Australia
        'AU_10Y': 'IRLTLT01AUM156N',

        # Brazil
        'BR_10Y': 'IRLTLT01BRM156N',

        # South Africa
        'ZA_10Y': 'IRLTLT01ZAM156N'
    }

    print(f"Downloading {len(fred_map)} FRED series...")

    fred_data = pd.DataFrame()
    for name, ticker in fred_map.items():
        try:
            s = web.DataReader(ticker, "fred", start, end)
            fred_data[name] = s[ticker]
        except Exception as e:
            print(f"Failed: {name} ({ticker})")

    # --------------------------------------------------------
    # 🔗 MERGE ALL
    # --------------------------------------------------------
    macro = pd.concat([yf_data, fred_data], axis=1)

    # Cleaning
    macro = macro.dropna(axis=1, how='all')
    macro = macro.ffill().dropna()

    print("Final macro shape:", macro.shape)

    return macro


# RUN
macro_data = get_macro_data_full()

# SAVE
macro_data.to_csv("macro_full.csv")
print("Saved: macro_full.csv")

/tmp/ipykernel_3399/213149508.py:37: FutureWarning: YF.download() has changed argument auto_adjust default to True
  yf_data = yf.download(yf_tickers, start=start, end=end)["Close"]
[*********************100%***********************]  13 of 13 completed


Failed: BR_10Y (IRLTLT01BRM156N)
Final macro shape: (4124, 19)
Saved: macro_full.csv
